# How Traders Actually Make Money: Edge and Expectancy

In the last lesson you learned that a quant trades by explicit rules. But a rule is only worth running if it makes money on average. This lesson makes that idea precise. You will stop thinking about trading as "being right" and start thinking about it the way professionals do: as a long sequence of bets with a small, measurable advantage.

By the end of this lesson you will be able to:

- Define what an edge is and why it must come from a real reason
- Compute expectancy from win rate, average win, and average loss
- Explain the tradeoff between hit rate and payoff ratio
- Show why a small positive expectancy, repeated many times, compounds into profit
- Account for how fees and slippage erode that edge

## 1. What An Edge Actually is ?

An edge is a reason the market will, on average, pay you for a particular action. That's it. Not certainty — average. If you flip a fair coin and win or lose a dollar, you have no edge: over many flips you break even. If someone pays you $1.05 when you win but you only lose $1.00 when you lose, you now have an edge, even though you still win only half the time.

In trading, an edge comes from one of a few sources: other participants are forced to trade (index funds rebalancing), react slowly to information (momentum), overreact and then correct (mean reversion), or demand a premium to hold risk (carry). If you cannot name why the market pays you, you probably have no edge — you have a curve that happened to fit the past.

>An edge is not a prediction. It is a statistical tilt that shows up only over many trades.

A useful sanity check is to ask: who is on the other side of my trade, and *why are they willing to lose to me on average?* Markets are roughly zero-sum before costs and negative-sum after them, so your gain is someone else's loss. If your edge is momentum, the other side might be a slow institution still digesting news. If it's mean reversion, the other side might be a panicked seller dumping at any price. If you genuinely cannot picture the loser on the other side, be suspicious — you may be the loser and not yet know it.

## 2. Expectancy : The one formula to internalize

Expectancy is the average profit or loss you expect per trade, given your win rate and the size of your wins and losses. The formula is:

```

expectancy = (win_rate * avg_win) - (loss_rate * avg_loss)

```

Here `lossrate = 1 - winrate`, `avgwin` is your average profit on winning trades (a positive number), and `avgloss` is your average loss on losing trades (entered as a positive number). If expectancy is positive, the strategy makes money on average. If it's zero or negative, no amount of discipline or position sizing will save it.

This single number reframes everything. You are not trying to be right. You are trying to keep expectancy positive and then take the bet as many times as you safely can.

It is worth seeing why the formula is just a `weighted average` in disguise. Over many trades, a fraction winrate of them earn `avgwin` and a fraction lossrate of them lose `avgloss`. The average outcome per trade is therefore the wins weighted by how often they happen minus the losses weighted by how often they happen — which is exactly the formula. Expectancy is nothing more mysterious than "average dollars per trade," written so you can see the two levers that control it.


## 3. A Numeric Worked Example

Suppose you build a strategy and, over 200 trades, you observe:

Win rate: 40% (you win only 4 trades out of 10)
Average win: $300
Average loss: $150
Plug it in:


```
expectancy = (0.40 * 300) - (0.60 * 150)
           = 120 - 90
           = 30

```

You make $30 per trade on average, even though you lose more often than you win. Over 200 trades that's roughly $6,000 — before costs. This is the crucial insight beginners miss: you can be wrong most of the time and still be highly profitable, as long as your winners are big enough relative to your losers.

Now flip it. Suppose a different strategy wins 70% of the time but its losses are large:

- Win rate: 70%, avg win: $100, avg loss: $260

```
expectancy = (0.70 * 100) - (0.30 * 260)
           = 70 - 78
           = -8

```
This "feels" great — you win 7 times out of 10 — but it loses $8 per trade. A high hit rate is seductive and frequently a trap.

## 4. Hit Rate Vs Payoff Ratio

Two levers drive expectancy, and they trade off against each other:

- Hit rate (win rate): how often you win.
- Payoff ratio: the ratio of average win to average loss (avgwin / avgloss).

You can build a profitable strategy at almost any hit rate, as long as the payoff ratio compensates. The breakeven relationship is simple: you are profitable when


```
win_rate > 1 / (1 + payoff_ratio)


```

|Payoff ratio (win/loss)	|Win rate needed to break even|
|-----|----|
|0.5|	66.7%|
|1.0|	50.0%|
|1.5|	40.0%|
|2.0|	33.3%|
|3.0|	25.0%|

Trend-following strategies typically sit in the bottom rows: they win maybe 35–40% of the time but let winners run far past losers. Mean-reversion strategies often sit near the top: high hit rate, small payoff ratio. Neither is better — they are just different points on the same curve.

The breakeven formula is worth deriving once so it never feels like magic. At breakeven, expectancy is exactly zero:


```
win_rate * avg_win = loss_rate * avg_loss

```

Divide both sides by avgloss and write `R = avgwin / avgloss` (the payoff ratio) and `lossrate = 1 - win_rate`:


```
win_rate * R = (1 - win_rate)
win_rate * R + win_rate = 1
win_rate * (R + 1) = 1
win_rate = 1 / (1 + R)

```

That's the whole table in one line of algebra. Any win rate above this line means positive expectancy; below it, you lose. Keep this relationship in your head — it lets you glance at a strategy's payoff ratio and immediately know the hit rate it must clear.



## 5. Why being right ~ 53% can be enough

If your payoff ratio is roughly 1 (wins and losses about the same size), the breakeven win rate is 50%. So a strategy that wins 53% of the time has a real, positive edge. That sounds tiny, and per trade it is. But you don't take the bet once

This is where the law of **large numbers does the work**. The law says that as you repeat a bet with positive expectancy, your average result converges toward the true expectancy, and the noise of any individual outcome washes out. One trade is a coin flip. A thousand trades with a 53% edge is, with very high probability, a profit.

Consider a 53/47 edge with even-sized bets of $100:

```
expectancy = (0.53 * 100) - (0.47 * 100) = 6

```

Six dollars per trade. Take it 2,000 times in a year and that's $12,000 of expected profit from an edge so small you could never feel it on any single trade. This is the entire game: find a small, durable edge and harvest it at scale, with risk control so a bad streak doesn't end you before the average arrives.

## 6. Worked Example: Simulating the law of large numbers

Words about "the average converges" are easy to nod along to and hard to feel. Let's make it tangible by simulating that 53/47 edge and watching what happens to the running average P&L as the number of trades grows.


In [1]:
import numpy as np

np.random.seed(42)

win_prob = 0.53
bet = 100
n_trades = 5000

## +100 on a win (prob 0.53), -100 on a loss (prob 0.47)
outcomes = np.where(np.random.random(n_trades) < win_prob , bet, -bet)

running_avg = np.cumsum(outcomes) / np.arange(1, n_trades + 1)

for n in [10, 50, 200, 1000, 5000]:
    print(f"after {n:5d} trades: avg P&L per trade = {running_avg[n-1]:6.2f}")

after    10 trades: avg P&L per trade = -20.00
after    50 trades: avg P&L per trade =  24.00
after   200 trades: avg P&L per trade =  12.00
after  1000 trades: avg P&L per trade =   7.60
after  5000 trades: avg P&L per trade =   5.84


If you run this, the early numbers are wild — after 10 trades the average might be +40 or -20, swinging all over the place. But as the trade count climbs, the running average settles down toward the true expectancy of +6. The individual trades never get less random; there are just so many of them that the noise averages out. That settling is the law of large numbers, and it is the only reason a 53% edge is worth anything at all.

Notice the flip side hidden in the early swings: with a tiny edge, short stretches can look terrible purely by chance. A trader who quits after 10 bad trades never reaches the regime where the edge pays. Survival long enough to let the average arrive is itself a skill — and the entire point of risk sizing later in the course.

## 7. Computing Expectancy from real trades in Python

In practice you don't know your win rate in advance — you measure it from a list of realized trade profits and losses. Here is a function that takes a list of per-trade P&Ls and returns the key statistics.

In [2]:
import numpy as np

def expectancy(pnls):
    pnls = np.asarray(pnls, dtype=float)
    n = len(pnls)
    if n == 0:
        return None

    wins = pnls[pnls > 0]
    losses = pnls[pnls < 0]

    win_rate = len(wins) / n
    loss_rate = len(losses) / n
    avg_win = wins.mean() if len(wins) else 0.0
    avg_loss = -losses.mean() if len(losses) else 0.0  # positive number

    exp = (win_rate * avg_win) - (loss_rate * avg_loss)
    payoff = (avg_win / avg_loss) if avg_loss > 0 else float("inf")

    return {
        "trades": n,
        "win_rate": round(win_rate, 4),
        "avg_win": round(avg_win, 2),
        "avg_loss": round(avg_loss, 2),
        "payoff_ratio": round(payoff, 2),
        "expectancy": round(exp, 2),
        "total_pnl": round(pnls.sum(), 2),
    }

trades = [300, -150, -150, 300, -150, 300, -150, 300, -150, -150]
print(expectancy(trades))

{'trades': 10, 'win_rate': 0.4, 'avg_win': np.float64(300.0), 'avg_loss': np.float64(150.0), 'payoff_ratio': np.float64(2.0), 'expectancy': np.float64(30.0), 'total_pnl': np.float64(300.0)}


Run it and you'll see an expectancy of $30 per trade, matching our hand calculation. Note that expectancy * trades equals total_pnl exactly — the formula is just the average P&L rewritten in terms of win rate and payoffs.

## 8. How fees and silppage erode the edge

Everything above ignored costs. Reality does not. Every trade pays a commission and loses something to slippage — the gap between the price you wanted and the price you got. These costs subtract directly from expectancy.

Suppose our $30 expectancy strategy costs $5 in commission and $8 in slippage per round trip:

```
net_expectancy = 30 - 5 - 8 = 17

```

Still positive — survivable. But now imagine the 53/47 even-money strategy with $6 of edge per trade. If costs are $7 per trade, your $6 edge becomes negative:

```
net_expectancy = 6 - 7 = -1

```

The strategy is mathematically sound and a guaranteed loser in practice. This is why high-frequency edges require ultra-low costs, and why trading more often is not automatically better. Costs scale with trade count; edge per trade does not. Always compute expectancy after realistic costs before believing a backtest.

## 9. Real-world case study: the same edge at two trade frequencies


Imagine two traders who have discovered the exact same underlying edge — a 55% directional hit rate on even-money bets, worth a gross expectancy of `(0.55 - 0.45) * 100 = $10` per trade on a $100 position. Round-trip costs are a flat $4 per trade for both.

Trader A trades it slowly: 100 times a year.

```
net per trade = 10 - 4 = 6
annual net    = 6 * 100 = 600

```

Trader B, excited by the edge, trades a faster version of it: 2,000 times a year. The gross edge per trade is the same $10, and so is the $4 cost.

```
net per trade = 10 - 4 = 6
annual net    = 6 * 2000 = 12,000

```
At first glance B looks twenty times better — same edge, far more trades. And if the cost truly stays at $4, B is better. But here is the catch that ruins most overtraders: a faster strategy almost always has a smaller gross edge per trade, because it is chasing finer, noisier moves. Suppose B's faster signal is only worth $5 gross per trade, not $10:

```
B net per trade = 5 - 4 = 1
B annual net     = 1 * 2000 = 2,000

```

B trades twenty times as much for a third of A's profit, and a small rise in costs would tip B into a loss while leaving A comfortably profitable. The moral: more trades only help if the edge per trade survives. Frequency multiplies both the edge and the costs, and costs are the part you can't wish away.

## 10. Common Mistakes

- Chasing win rate. A 90% win rate with occasional catastrophic losses is a path to ruin. Expectancy, not hit rate, is what matters.
- Ignoring costs until live trading. A pre-cost edge that's smaller than your per-trade costs is not an edge.
- Confusing one trade with the edge. A losing trade does not mean the edge is gone; a winning trade does not prove it exists. Only large samples reveal expectancy.
- Sizing too large. Even a positive-expectancy strategy can be wiped out by a losing streak if each bet is too big. Edge needs survival to compound; we cover sizing in the risk module.
- No reason behind the edge. A positive backtest expectancy with no economic explanation is usually overfitting.